# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [ ]:
# Load the libraries as required.

In [3]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [4]:
import pandas as pd

# Define file path
file_path = '../../05_src/data/fires/forestfires.csv'

# Define column names
columns = ['coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area']

# Load dataset
fires_dt = pd.read_csv(file_path, header=0, names=columns)

# Separate features (X) and target (y)
X = fires_dt.drop(columns=['area'])  # Drop target column
y = fires_dt['area']  # Target variable

# Print shapes to verify
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (517, 12)
y shape: (517,)


In [5]:
# Convert categorical variables (month, day) to numerical using one-hot encoding
X = pd.get_dummies(X, columns=['month', 'day'], drop_first=True)

# Print the transformed features
print(X.head())  # View the first few rows

   coord_x  coord_y  ffmc   dmc     dc  isi  temp  rh  wind  rain  ...  \
0        7        5  86.2  26.2   94.3  5.1   8.2  51   6.7   0.0  ...   
1        7        4  90.6  35.4  669.1  6.7  18.0  33   0.9   0.0  ...   
2        7        4  90.6  43.7  686.9  6.7  14.6  33   1.3   0.0  ...   
3        8        6  91.7  33.3   77.5  9.0   8.3  97   4.0   0.2  ...   
4        8        6  89.3  51.3  102.2  9.6  11.4  99   1.8   0.0  ...   

   month_may  month_nov  month_oct  month_sep  day_mon  day_sat  day_sun  \
0      False      False      False      False    False    False    False   
1      False      False       True      False    False    False    False   
2      False      False       True      False    False     True    False   
3      False      False      False      False    False    False    False   
4      False      False      False      False    False    False     True   

   day_thu  day_tue  day_wed  
0    False    False    False  
1    False     True    False  
2    

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [6]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Load dataset
file_path = '../../05_src/data/fires/forestfires.csv'
columns = ['coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area']
fires_dt = pd.read_csv(file_path, header=0, names=columns)

# Define X (features) and y (target)
X = fires_dt.drop(columns=['area'])
y = fires_dt['area']

# Identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# Create a preprocessing pipeline for numerical features
num_transforms = Pipeline(steps=[
    ('scaler', StandardScaler())  # Scaling numeric features
])

# Create a preprocessing pipeline for categorical features
cat_transforms = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))  # One-hot encoding categorical features
])

# Create the Column Transformer (Preproc1)
preproc1 = ColumnTransformer(
    transformers=[
        ('num', num_transforms, numerical_cols), 
        ('cat', cat_transforms, categorical_cols) 
    ]
)

# Fit and transform the data
X_transformed = preproc1.fit_transform(X)

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [7]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.pipeline import Pipeline

# Load dataset
file_path = '../../05_src/data/fires/forestfires.csv'
columns = ['coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area']
fires_dt = pd.read_csv(file_path, header=0, names=columns)

# Define X (features) and y (target)
X = fires_dt.drop(columns=['area'])
y = fires_dt['area']

# Identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# Create a preprocessing pipeline for numerical features
num_transforms = Pipeline(steps=[
    ('scaler', StandardScaler()),         # Scaling numeric features
    ('transform', PowerTransformer())     # Applying non-linear transformation (e.g., Yeo-Johnson)
])

# Create a preprocessing pipeline for categorical features
cat_transforms = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))  # One-hot encoding categorical features
])

# Create the Column Transformer (Preproc2)
preproc2 = ColumnTransformer(
    transformers=[
        ('num', num_transforms, numerical_cols), 
        ('cat', cat_transforms, categorical_cols) 
    ]
)

# Fit and transform the data
X_transformed = preproc2.fit_transform(X)

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [13]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Define the baseline model
baseline_model = Ridge(alpha=1.0)  # Example: Ridge regression model

# Define the advanced model
advanced_model = RandomForestRegressor(n_estimators=100, random_state=42)  # Example: Random Forest model

# Create the Model Pipeline
pipeline_a = Pipeline([
    ('preprocessing', preproc1),   # Preprocessing: Preproc1 (scaling only)
    ('regressor', baseline_model)  # Regressor: Baseline Model (Ridge)
])

pipeline_b = Pipeline([
    ('preprocessing', preproc2),   # Preprocessing: Preproc2 (scaling + non-linear transform)
    ('regressor', baseline_model)  # Regressor: Baseline Model (Ridge)
])

pipeline_c = Pipeline([
    ('preprocessing', preproc1),   # Preprocessing: Preproc1 (scaling only)
    ('regressor', advanced_model)  # Regressor: Advanced Model (Random Forest)
])

pipeline_d = Pipeline([
    ('preprocessing', preproc2),   # Preprocessing: Preproc2 (scaling + non-linear transform)
    ('regressor', advanced_model)  # Regressor: Advanced Model (Random Forest)
])


from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Define the baseline model (Ridge regression)
baseline_model = Ridge(alpha=1.0)  # Example: Ridge regression model

# Define the advanced model (Random Forest)
advanced_model = RandomForestRegressor(n_estimators=100, random_state=42)  # Example: Random Forest model


In [14]:
# Pipeline A = preproc1 + baseline

pipeline_a = Pipeline([
    ('preprocessing', preproc1),   # Preprocessing: Preproc1 (scaling only)
    ('regressor', baseline_model)  # Regressor: Baseline Model (Ridge)
])


In [15]:
# Pipeline B = preproc2 + baseline

pipeline_b = Pipeline([
    ('preprocessing', preproc2),   # Preprocessing: Preproc2 (scaling + non-linear transform)
    ('regressor', baseline_model)  # Regressor: Baseline Model (Ridge)
])


In [17]:
# Pipeline C = preproc1 + advanced model

pipeline_c = Pipeline([
    ('preprocessing', preproc1),   # Preprocessing: Preproc1 (scaling only)
    ('regressor', advanced_model)  # Regressor: Advanced Model (Random Forest)
])




In [18]:
# Pipeline D = preproc2 + advanced model

pipeline_d = Pipeline([
    ('preprocessing', preproc2),   # Preprocessing: Preproc2 (scaling + non-linear transform)
    ('regressor', advanced_model)  # Regressor: Advanced Model (Random Forest)
])



# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [19]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [20]:
# Hyperparameter grid for Ridge (Pipeline A and B)
param_grid_ridge = {
    'regressor__alpha': [0.1, 1.0, 10.0, 100.0]  # Hyperparameter for Ridge
}

# Hyperparameter grid for RandomForestRegressor (Pipeline C and D)
param_grid_rf = {
    'regressor__n_estimators': [50, 100, 200, 300],   # Number of trees
    'regressor__max_depth': [None, 10, 20, 30]  # Depth of each tree
}


In [21]:
# GridSearchCV for Pipeline A (Ridge with preproc1)
grid_a = GridSearchCV(estimator=pipeline_a, param_grid=param_grid_ridge, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# GridSearchCV for Pipeline B (Ridge with preproc2)
grid_b = GridSearchCV(estimator=pipeline_b, param_grid=param_grid_ridge, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# GridSearchCV for Pipeline C (RandomForest with preproc1)
grid_c = GridSearchCV(estimator=pipeline_c, param_grid=param_grid_rf, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# GridSearchCV for Pipeline D (RandomForest with preproc2)
grid_d = GridSearchCV(estimator=pipeline_d, param_grid=param_grid_rf, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)


In [23]:
from sklearn.model_selection import train_test_split

# Assuming 'X' contains the features and 'y' contains the target variable (area)
X = fires_dt.drop('area', axis=1)
y = fires_dt['area']

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Fit GridSearchCV for each pipeline
grid_a.fit(X_train, y_train)  # Fit GridSearch for Pipeline A
grid_b.fit(X_train, y_train)  # Fit GridSearch for Pipeline B
grid_c.fit(X_train, y_train)  # Fit GridSearch for Pipeline C
grid_d.fit(X_train, y_train)  # Fit GridSearch for Pipeline D


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler()),
                                                                                         ('transform',
                                                                                          PowerTransformer())]),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor',
                                        RandomForestRegressor(random_state=42))]),
             n_jobs=-1,
             param_grid={'regressor__max_depth': [None, 10, 20, 30],
                         'regressor__n_estimators': [50, 100, 200, 300]},
             scoring='neg_mean_squared_error')

In [24]:
# Evaluate the best model for each pipeline
best_model_a = grid_a.best_estimator_
best_model_b = grid_b.best_estimator_
best_model_c = grid_c.best_estimator_
best_model_d = grid_d.best_estimator_

# Print best hyperparameters
print("Best hyperparameters for Pipeline A:", grid_a.best_params_)
print("Best hyperparameters for Pipeline B:", grid_b.best_params_)
print("Best hyperparameters for Pipeline C:", grid_c.best_params_)
print("Best hyperparameters for Pipeline D:", grid_d.best_params_)

# Predict and evaluate performance for each model
y_pred_a = best_model_a.predict(X_test)
y_pred_b = best_model_b.predict(X_test)
y_pred_c = best_model_c.predict(X_test)
y_pred_d = best_model_d.predict(X_test)

# Evaluate each model using mean absolute error, mean squared error, and R-squared
def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, r2

# Evaluation for each pipeline
mae_a, mse_a, r2_a = evaluate_model(y_test, y_pred_a)
mae_b, mse_b, r2_b = evaluate_model(y_test, y_pred_b)
mae_c, mse_c, r2_c = evaluate_model(y_test, y_pred_c)
mae_d, mse_d, r2_d = evaluate_model(y_test, y_pred_d)

# Print performance metrics
print("Pipeline A - MAE:", mae_a, "MSE:", mse_a, "R2:", r2_a)
print("Pipeline B - MAE:", mae_b, "MSE:", mse_b, "R2:", r2_b)
print("Pipeline C - MAE:", mae_c, "MSE:", mse_c, "R2:", r2_c)
print("Pipeline D - MAE:", mae_d, "MSE:", mse_d, "R2:", r2_d)


Best hyperparameters for Pipeline A: {'regressor__alpha': 100.0}
Best hyperparameters for Pipeline B: {'regressor__alpha': 100.0}
Best hyperparameters for Pipeline C: {'regressor__max_depth': None, 'regressor__n_estimators': 100}
Best hyperparameters for Pipeline D: {'regressor__max_depth': None, 'regressor__n_estimators': 100}
Pipeline A - MAE: 24.177820664703503 MSE: 11740.027150130672 R2: 0.004049536297947243
Pipeline B - MAE: 23.964291297255404 MSE: 11714.462177314932 R2: 0.006218308670001171
Pipeline C - MAE: 26.970422525183153 MSE: 12026.536440864651 R2: -0.020256119669660677
Pipeline D - MAE: 26.733182380952385 MSE: 12050.84692139589 R2: -0.022318468763771415


# Evaluate

+ Which model has the best performance?

Pipeline B (Preproc2 + Baseline Regressor) has the best performance with the lowest MAE (23.96) and the highest R2 (0.0062). Its hyperparameters are alpha = 100.0 (Ridge Regressor). Pipeline A also performed similarly but with slightly higher MAE and R2 values. Overall, Pipeline B offers the best model balance, achieving reasonable error reduction without overfitting.

# Export

+ Save the best performing model to a pickle file.

In [25]:
import pickle

# Assuming best_model_b is the best performing model from Pipeline B
best_model_b = grid_b.best_estimator_

# Save the best model to a pickle file
with open('best_model_b.pkl', 'wb') as f:
    pickle.dump(best_model_b, f)
    
print("Best model saved successfully to 'best_model_b.pkl'")


Best model saved successfully to 'best_model_b.pkl'


# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

If we were to remove features from the model, we would prioritize eliminating those with the least influence on the outcome, as indicated by their SHAP importance. To test if these features are truly enhancing model performance, we would compare the model’s performance metrics (e.g., accuracy, RMSE) before and after the removal of these features, ensuring that removing them does not significantly degrade the model’s predictive power.

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.